**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/costabile-2023"], check=True)
    os.chdir("/content/encore/entries/costabile-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Costabile et al. (2023): MINE and the linear model on a random subsample of components

**Paper.** Costabile JD, Balakrishnan KA, Schwinn S, Haesemeyer M (2023). Model discovery to link
neural activity to behavioral tasks. *eLife* 12:e83289. [doi:10.7554/eLife.83289](https://doi.org/10.7554/eLife.83289)

**Datasets.** DANDI:000235 (0.230316.1600), 000236 (0.230316.2031) and 000237 (0.230316.1655), as in
`reproduce_costabile_2023.ipynb`, which has the header, the data checks and the summary.

**Why this notebook.** The claims on the MINE fits need the paper's CNN fit to every one of the
432,882 bright components, which does not fit in a run (`reproduce_costabile_2023.ipynb`,
section 5, and the timing at the end of this notebook). The round-1 verifier asked for what can be
tested at small cost: the linear comparison model on every component (the main notebook, section 5)
and MINE on a random subsample. This notebook fits MINE to 20 random bright components in each of 30
random planes (600 fits) and estimates, from them, the quantities of fig6d-mine-identifies-more-than-lm
and figs6-mine-more-at-every-cutoff. The estimates decide no outcome: those claims stay blocked as
too large, and the notes give these numbers as estimates.

**Method.** As the claims file's opening comment and rwave_data_fit.py at mine_pub 368d756 (see
`mine_cnn.py`): CaImAn's calcium on the 5 Hz base, z-scored and smoothed (sigma 2.5 frames);
temperature and the three behavior predictors (bouts under the main reading, thr5_min10; see the main
notebook, section 1), the behavior mean-subtracted, Gram-Schmidt orthogonalized and z-scored; the
CNN of get_standard_model, 100 epochs on the first two-thirds; the linear model a Ridge regression
(alpha 1e-4) on the orthonormalized time-shifted predictors. A neuron is identified at a test
correlation of at least sqrt(0.5) (claims file).

**Sampling and seeds.** Seeds from the claim, `zlib.crc32(b"costabile-2023/fig6d-mine-identifies-more-than-lm")`:
the first generator draws the planes, the second (seed + 1) the components in each plane, the third
(seed + 2) one seed for each fit, the fourth (seed + 3) the bootstrap over planes. Totals are
estimated plane by plane (each plane's bright count times its sampled fraction) and scaled by 750/30;
the 95% intervals come from 2,000 bootstrap resamples of the 30 planes. The protocol's repeat under
four more seeds would take four more sets of 600 fits (about 45 minutes here, at the wall time printed in section 2), so it is not run; the
bootstrap gives the sampling error instead, and no outcome rests on these numbers.

In [ ]:
import os, time, zlib
os.environ.setdefault("OMP_NUM_THREADS", "1")
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import mine_data as md
import mine_cnn
T0 = time.time()
SEED = zlib.crc32(b"costabile-2023/fig6d-mine-identifies-more-than-lm")
CUT = np.sqrt(0.5)
N_PLANES, N_PER_PLANE = 30, 20
if os.environ.get("COSTABILE_SUBSET"):  # development only
    N_PLANES, N_PER_PLANE = 3, 2
print("CPUs:", os.cpu_count(), "; usable by this process:", len(os.sched_getaffinity(0)))

## 1. The sample

In [ ]:
allp = []
for d in md.ZEBRAFISH_WHOLE_BRAIN:
    for a in md.list_assets(d):
        with md.open_nwb(a["asset_id"]) as f:
            allp += [(a["asset_id"], d, p) for p in md.planes(f)]
print(len(allp), "whole-brain planes")
rng1 = np.random.default_rng(SEED)
pick = sorted(rng1.choice(len(allp), N_PLANES, replace=False))
tasks = [allp[i] for i in pick]
t = time.time()
with ProcessPoolExecutor(4) as ex:
    res = list(ex.map(md.plane_pass, *zip(*[(a, d, p, True) for a, d, p in tasks])))
print(f"plane pass {time.time() - t:.0f} s")
rng2 = np.random.default_rng(SEED + 1)
rng3 = np.random.default_rng(SEED + 2)
samples = []
for r in res:
    bright = np.nonzero(r["brightness"] >= md.BRIGHTNESS_CUT)[0]
    comps = rng2.choice(bright, min(N_PER_PLANE, bright.size), replace=False)
    samples.append(dict(comps=comps, seeds=rng3.integers(0, 2**31 - 1, comps.size), n_bright=bright.size))
sp = pd.DataFrame([dict(dandiset=r["dandiset"], fish=r["fish"].split("_")[0], plane=r["plane"], n_rois=r["n_rois"],
                        n_bright=s["n_bright"], sampled=s["comps"].size) for r, s in zip(res, samples)])
print(sp.groupby("dandiset").agg(planes=("plane", "size"), fish=("fish", "nunique"), bright=("n_bright", "sum"),
                                 sampled=("sampled", "sum")).to_string())
print("fish represented:", sp.fish.nunique(), "of 25; components sampled:", sp.sampled.sum())

## 2. The fits

In [ ]:
t = time.time()
with ProcessPoolExecutor(4) as ex:  # one task per plane
    futs = [ex.submit(mine_cnn.fit_plane_components, a, p, s["comps"], r["temp5"], r["behavior"][md.PRIMARY_BOUTS],
                      s["seeds"], r["frame_duration"]) for (a, d, p), r, s in zip(tasks, res, samples)]
    fits = [fu.result() for fu in futs]
wall = time.time() - t
ft = np.concatenate([f["t"] for f in fits])
n_fit = ft.size
mine_r = [f["r"] for f in fits]
lm_r = [r["lm_r"][md.PRIMARY_BOUTS][s["comps"]] for r, s in zip(res, samples)]
print(f"{n_fit} fits in {wall/60:.1f} min on 4 processes: {wall/n_fit:.2f} s of wall time per fit; "
      f"time of one fit in a process median {np.median(ft):.2f} s (5th to 95th percentile {np.percentile(ft, 5):.2f} to {np.percentile(ft, 95):.2f})")
mr, lr = np.concatenate(mine_r), np.concatenate(lm_r)
print("fits with an undefined test correlation:", int(np.isnan(mr).sum()), "MINE,", int(np.isnan(lr).sum()), "linear model")
print(f"test correlation, MINE median {np.nanmedian(mr):.3f} (range {np.nanmin(mr):.3f} to {np.nanmax(mr):.3f}); "
      f"linear model median {np.nanmedian(lr):.3f} (range {np.nanmin(lr):.3f} to {np.nanmax(lr):.3f})")
print(f"MINE above the linear model in {np.mean(mr > lr):.1%} of sampled components")

## 3. fig6d-mine-identifies-more-than-lm, estimated

> Of the 432,882 active neurons, about 42,000 are identified by MINE or by the linear comparison
> model; 40% only by MINE, 54% by both and 6% only by the linear model.

Estimates from the subsample (decide no outcome; the claim stays blocked as too large). The code's
own counting threshold (0.6, printed in rwave_data_fit.py) is shown as a variant.

In [ ]:
scale = len(allp) / N_PLANES
def totals(idx, cut):
    out = np.zeros(4)  # mine only, both, lm only, sampled-bright total
    for i in idx:
        m, l = mine_r[i] >= cut, lm_r[i] >= cut
        w = samples[i]["n_bright"] / m.size
        out += w * np.array([np.sum(m & ~l), np.sum(m & l), np.sum(~m & l), m.size])
    return out * scale
rng4 = np.random.default_rng(SEED + 3)
boots = [rng4.integers(0, N_PLANES, N_PLANES) for _ in range(2000)]
rows = []
for cut in (CUT, 0.6):
    est = totals(range(N_PLANES), cut)
    B = np.array([totals(b, cut) for b in boots])
    def row(name, f):
        v = f(est); bv = np.array([f(x) for x in B])
        return dict(cut=round(cut, 3), quantity=name, estimate=v, lo=np.percentile(bv, 2.5), hi=np.percentile(bv, 97.5))
    rows += [row("bright components", lambda x: x[3]),
             row("MINE or linear model", lambda x: x[:3].sum()),
             row("MINE (all)", lambda x: x[0] + x[1]),
             row("linear model (all)", lambda x: x[1] + x[2]),
             row("fraction only MINE", lambda x: x[0] / x[:3].sum()),
             row("fraction both", lambda x: x[1] / x[:3].sum()),
             row("fraction only linear model", lambda x: x[2] / x[:3].sum())]
tab = pd.DataFrame(rows)
print(tab.to_string(index=False, float_format=lambda v: f"{v:.3f}" if abs(v) < 2 else f"{v:,.0f}"))
n_sel = [int(np.sum((mine_r[i] >= CUT) | (lm_r[i] >= CUT))) for i in range(N_PLANES)]
print("sampled components identified by either model at sqrt(0.5):", sum(n_sel), "; by MINE:",
      int(sum(np.sum(m >= CUT) for m in mine_r)), "; by the linear model:", int(sum(np.sum(l >= CUT) for l in lm_r)))
print("expected: union 42,229 (tolerance 4,223); fractions 0.40 / 0.54 / 0.06 (tolerances 0.04 / 0.054 / 0.006)")

In [ ]:
# Per dandiset, at sqrt(0.5): sampled components and the share each model identifies.
pdf = pd.DataFrame([dict(dandiset=sp.dandiset[i], fish=sp.fish[i], n=mine_r[i].size, mine=np.sum(mine_r[i] >= CUT),
                         lm=np.sum(lm_r[i] >= CUT), both=np.sum((mine_r[i] >= CUT) & (lm_r[i] >= CUT))) for i in range(N_PLANES)])
g = pdf.groupby("dandiset")[["n", "mine", "lm", "both"]].sum()
g["mine_frac"], g["lm_frac"] = g.mine / g.n, g.lm / g.n
print(g.to_string(float_format=lambda v: f"{v:.3f}"))
print("fish with at least one identified sampled component:", pdf[pdf.mine + pdf.lm > 0].fish.nunique(), "of", pdf.fish.nunique())

## 4. figs6-mine-more-at-every-cutoff, estimated

> MINE identifies more neurons than the linear model whatever test-correlation cutoff is used.

Estimated counts at each cutoff from 0 to 1 in steps of 0.01 (rwave_plot_panels.py), from the
subsample, with the bootstrap interval of the difference.

In [ ]:
cuts = np.round(np.arange(0, 1.001, 0.01), 2)
def counts(idx, cut):
    m = l = 0.0
    for i in idx:
        w = samples[i]["n_bright"] / mine_r[i].size
        m += w * np.sum(mine_r[i] >= cut); l += w * np.sum(lm_r[i] >= cut)
    return m * scale, l * scale
cur = pd.DataFrame([dict(cut=c, mine=counts(range(N_PLANES), c)[0], lm=counts(range(N_PLANES), c)[1]) for c in cuts])
D = np.array([[np.subtract(*counts(b, c)) for c in cuts] for b in boots[:500]])
cur["diff_lo"], cur["diff_hi"] = np.percentile(D, 2.5, 0), np.percentile(D, 97.5, 0)
show = cur[(cur.mine > 0) | (cur.lm > 0)]
print(show.iloc[::5].to_string(index=False, float_format=lambda v: f"{v:,.0f}" if abs(v) >= 2 else f"{v:.2f}"))
bad = show[show.mine < show.lm]
print("cutoffs with a count above zero:", len(show), "; where the MINE estimate is below the linear model's:",
      bad.cut.tolist() if len(bad) else "none")
print("cutoffs where the interval of MINE minus linear model lies wholly below 0:", show[show.diff_hi < 0].cut.tolist() or "none")
print("highest cutoff with a sampled component: MINE", float(np.nanmax(mr)).__round__(3), "; linear model", float(np.nanmax(lr)).__round__(3))

## 5. What the full analysis would take

In [ ]:
per_fit_wall = wall / n_fit
for name, n in [("real data, 432,882 fits", 432882), ("with the rotated controls", 2 * 432882),
                ("backfill fish of 000238, 218,257 components", 218257)]:
    print(f"{name}: {n * per_fit_wall / 3600:.0f} h on this machine's 4 CPUs at {per_fit_wall:.2f} s of wall time per fit")
print(f"notebook elapsed: {(time.time() - T0)/60:.1f} min")

**Result: estimates only, no outcome.** From 600 fits (20 random bright components in each of 30
random planes, 16 of the 25 fish), MINE identifies 45 sampled components at sqrt(0.5) and the linear
model 30. Scaled to the dataset, MINE or the linear model identify about 54,000 neurons (95%
interval over planes 27,000 to 87,000; the paper 42,229), 37% only by MINE (23 to 61%; the paper
40%), 51% by both (30 to 64%; the paper 54%) and 11% only by the linear model (5 to 19%; the paper
6%). The paper's values all lie inside these intervals, which are far wider than the claim's
tolerances. MINE's estimated count is above the linear model's at every cutoff from 0.05 to 0.95,
and below it at cutoff 0. figs6-mine-more-at-every-cutoff asks for MINE above the linear model at
every cutoff where either count is above zero, so the point estimate fails at 0, although the
interval of the difference there includes zero and no interval lies wholly below it. Seven of the
600 MINE fits gave a constant prediction and no test correlation; they count as not identified at
every cutoff, which probably accounts for part of the gap at 0. In four processes a fit is slower
than alone on one thread; the time of one fit, the wall time per fit and the hours the full
analysis would take on this machine are printed in sections 2 and 5 (the timing varies between runs
by tens of percent), and they are many times the five hours of a run, twice as many with the
rotated controls.

## Summary

This notebook decides no outcome; the claims on the fits are blocked as too large (the attempt record
and the summary of `reproduce_costabile_2023.ipynb`). It differs from the paper's method in fitting
a random subsample of 600 bright components (20 in each of 30 random planes) and in using bouts
re-derived from the tail tracking, as the main notebook does.